# Notebook 4: The Virtual Knowledge Graph (Ontop)

The VKG pattern uses Ontop to expose a relational database as a SPARQL endpoint - no ETL, no data duplication. Ontop translates SPARQL to SQL on the fly.


---
### Setup


In [ ]:
ONTOP_ENDPOINT = ""  # Set by lifecycle config

In [ ]:
from workshop import *
configure(ontop_endpoint=ONTOP_ENDPOINT)

---
## Module 6: Querying the Virtual Knowledge Graph (Ontop)

The **Virtual Knowledge Graph** (VKG) pattern uses Ontop to expose a relational database (H2) as a SPARQL endpoint - no ETL, no data duplication. Ontop translates SPARQL queries to SQL on the fly using OWL ontology mappings (OBDA).

```
SPARQL Query → Ontop → SQL (H2) → Relational Results → RDF Response
```

### OBDA Mappings

The VKG is defined by 8 mappings across 4 relational tables. Each query below exercises a different combination:

| Mapping | Table | What It Virtualizes |
|---------|-------|---------------------|
| `policy` | `policy` | `:Policy` instances with `:policyId` |
| `policy_coverage` | `policy_coverage` | `:PolicyCoverage` - amount, deductible, premium, validFrom, isCurrent |
| `policy_coverage_validTo` | `policy_coverage` | Optional `:validTo` (only where non-NULL) |
| `policy_hasCoverage` | `policy_coverage` | `:Policy → :hasCoverage → :PolicyCoverage` link |
| `claim` | `claim` | `:Claim` - claimId, amount, incidentDate, `:forPolicy` link |
| `claim_event` | `claim_event` | `:ClaimEvent` - eventType, eventTimestamp |
| `claim_event_amount` | `claim_event` | Optional `:eventAmount` (only where non-NULL) |
| `claim_hasEvent` | `claim_event` | `:Claim → :hasEvent → :ClaimEvent` link |

The following queries show how SPARQL patterns map to SQL through these OBDA definitions.

In [ ]:
# ── 6.1: Single-Entity Lookup ────────────────────────────────────────
# Mappings: claim, claim_event, claim_event_amount, claim_hasEvent
# Ontop joins claim + claim_event via SQL to resolve :hasEvent
#
# "Get claim C003 and its event timeline"

print(f"Querying Ontop VKG at {ONTOP_URL}\n")
print("6.1  Single-Entity Lookup — Claim C003 event timeline")
print("     Mappings: claim, claim_event, claim_event_amount, claim_hasEvent\n")

q1 = """
PREFIX : <http://example.org/insurance#>
SELECT ?claimId ?claimAmount ?incidentDate ?eventType ?eventTimestamp ?eventAmount
WHERE {
    ?claim a :Claim ; :claimId "C003" ;
           :claimAmount ?claimAmount ; :incidentDate ?incidentDate ;
           :hasEvent ?event .
    ?event :eventType ?eventType ; :eventTimestamp ?eventTimestamp .
    OPTIONAL { ?event :eventAmount ?eventAmount }
}
ORDER BY ?eventTimestamp
"""

rows = ontop_query(q1)
print(f"  Claim C003 — ${float(rows[0]['claimAmount']):,.2f} — Incident {rows[0]['incidentDate'][:10]}\n")
print(f"  {'Event':<12} {'Timestamp':<22} {'Amount':>10}")
print(f"  {'─' * 48}")
for r in rows:
    amt = f"${float(r['eventAmount']):>9,.2f}" if r.get('eventAmount') else f"{'—':>10}"
    print(f"  {r['eventType']:<12} {r['eventTimestamp'][:19]:<22} {amt}")


### Query 2: Cross-Entity Temporal Join

This is the most revealing query. Ontop translates the SPARQL temporal FILTER into a SQL `WHERE` clause that finds coverage active at each claim's incident date. Notice that **Claim C003 is missing** from the results - no coverage row satisfies the temporal condition during the gap period. This is the same gap that SHACL detected in Notebook 02, but discovered here through query results rather than validation:


In [ ]:
# ── 6.2: Cross-Entity Temporal Join ──────────────────────────────────
# Mappings: claim, policy, policy_hasCoverage, policy_coverage, policy_coverage_validTo
# Ontop translates the temporal FILTER into a SQL WHERE clause:
#   WHERE valid_from <= incident_date AND (valid_to IS NULL OR valid_to > incident_date)
#
# "For each claim, show the active coverage at incident date"

print("6.2  Cross-Entity Temporal Join — Active coverage at incident date")
print("     Mappings: claim, policy, policy_hasCoverage, policy_coverage, policy_coverage_validTo\n")

q2 = """
PREFIX : <http://example.org/insurance#>
SELECT ?claimId ?claimAmount ?policyId ?coverageAmount ?deductible
WHERE {
    ?claim a :Claim ; :claimId ?claimId ; :claimAmount ?claimAmount ;
           :incidentDate ?incidentDate ; :forPolicy ?policy .
    ?policy :policyId ?policyId ; :hasCoverage ?cov .
    ?cov :coverageAmount ?coverageAmount ; :deductible ?deductible ;
         :validFrom ?vf .
    OPTIONAL { ?cov :validTo ?vt }
    FILTER (?vf <= ?incidentDate && (!BOUND(?vt) || ?vt > ?incidentDate))
}
ORDER BY ?claimId
"""

rows = ontop_query(q2)
print(f"  {'Claim':<8} {'Amount':>10} {'Policy':<8} {'Coverage':>12} {'Deductible':>12}")
print(f"  {'─' * 56}")
for r in rows:
    print(f"  {r['claimId']:<8} ${float(r['claimAmount']):>9,.2f} {r['policyId']:<8} ${float(r['coverageAmount']):>11,.2f} ${float(r['deductible']):>11,.2f}")

print(f"\n  Note: C003 is missing — no active coverage at its incident date (the coverage gap).")


### Query 3: Aggregation

Ontop pushes `GROUP BY` and `SUM` down into a single SQL aggregation query. This shows total claim exposure per policy - useful for risk assessment. The SPARQL is identical to what you'd write against Neptune; Ontop translates it to efficient SQL:


In [ ]:
# ── 6.3: Aggregation — Total Claim Exposure per Policy ───────────────
# Mappings: claim, policy
# Ontop pushes GROUP BY + SUM into a single SQL aggregation query
#
# "How much total claim exposure does each policy carry?"

print("6.3  Aggregation — Total claim exposure per policy")
print("     Mappings: claim, policy\n")

q3 = """
PREFIX : <http://example.org/insurance#>
SELECT ?policyId (COUNT(?claim) AS ?claimCount) (SUM(?claimAmount) AS ?totalExposure)
WHERE {
    ?claim a :Claim ; :claimAmount ?claimAmount ; :forPolicy ?policy .
    ?policy :policyId ?policyId .
}
GROUP BY ?policyId
ORDER BY DESC(?totalExposure)
"""

rows = ontop_query(q3)
print(f"  {'Policy':<10} {'Claims':>8} {'Total Exposure':>16}")
print(f"  {'─' * 38}")
for r in rows:
    print(f"  {r['policyId']:<10} {r['claimCount']:>8} ${float(r['totalExposure']):>14,.2f}")


### Query 4: SCD2 Coverage History

This query retrieves all coverage versions for every policy, showing the temporal dimension of the data. Ontop translates the `OPTIONAL { :validTo }` into a SQL `LEFT JOIN`. Look for the gap in Policy P002 - coverage v1 ends 2025-12-31, coverage v2 starts 2026-02-01:


In [ ]:
# ── 6.4: SCD2 Coverage History — Temporal Versions with Gap Detection ──
# Mappings: policy, policy_hasCoverage, policy_coverage, policy_coverage_validTo
# Ontop translates OPTIONAL { :validTo } into a LEFT JOIN on the validTo column
# and the isCurrent flag maps directly to the is_current SQL column.
#
# "Show all coverage versions for every policy, highlighting temporal gaps"

print("6.4  SCD2 Coverage History — All versions with gap detection")
print("     Mappings: policy, policy_hasCoverage, policy_coverage, policy_coverage_validTo\n")

q4 = """
PREFIX : <http://example.org/insurance#>
SELECT ?policyId ?coverageAmount ?deductible ?premium
       ?validFrom ?validTo ?isCurrent
WHERE {
    ?policy a :Policy ; :policyId ?policyId ; :hasCoverage ?cov .
    ?cov :coverageAmount ?coverageAmount ;
         :deductible ?deductible ;
         :premium ?premium ;
         :validFrom ?validFrom ;
         :isCurrent ?isCurrent .
    OPTIONAL { ?cov :validTo ?validTo }
}
ORDER BY ?policyId ?validFrom
"""

rows = ontop_query(q4)
current_policy = None
for r in rows:
    if r['policyId'] != current_policy:
        current_policy = r['policyId']
        print(f"\n  Policy {current_policy}")
        print(f"  {'Coverage':>12} {'Deductible':>12} {'Premium':>10} {'Valid From':<12} {'Valid To':<12} {'Status'}")
        print(f"  {'─' * 78}")
    vt = r['validTo'][:10] if r.get('validTo') else 'present'
    status = 'CURRENT' if r['isCurrent'] in ('true', 'True', True) else 'EXPIRED'
    print(f"  ${float(r['coverageAmount']):>11,.2f} ${float(r['deductible']):>11,.2f} ${float(r['premium']):>9,.2f} {r['validFrom'][:10]:<12} {vt:<12} {status}")


### Query 5: Full Pipeline Join

The most complex query - it joins across all 8 OBDA mappings to show denied claims with their full event timeline and coverage context. Ontop generates a multi-table SQL JOIN across claim, claim_event, policy, and policy_coverage. This is the complete picture an agent would need to explain a denial:


In [ ]:
# ── 6.5: Full Pipeline Join — Denied Claims with Event + Coverage Context ──
# Mappings: ALL 8 — claim, claim_event, claim_event_amount, claim_hasEvent,
#           policy, policy_hasCoverage, policy_coverage, policy_coverage_validTo
# Ontop generates a multi-table SQL JOIN across claim, claim_event,
# policy, and policy_coverage — the full relational model in one SPARQL query.
#
# "Show denied claims with their denial date and the coverage that was
#  active (or missing) at the time of the incident"

print("6.5  Full Pipeline Join — Denied claims with event + coverage context")
print("     Mappings: ALL 8 (claim, claim_event, claim_event_amount, claim_hasEvent,")
print("                      policy, policy_hasCoverage, policy_coverage, policy_coverage_validTo)\n")

q5 = """
PREFIX : <http://example.org/insurance#>
SELECT ?claimId ?claimAmount ?incidentDate ?policyId
       ?deniedDate ?coverageAmount ?deductible ?covValidFrom ?covValidTo
WHERE {
    ?claim a :Claim ; :claimId ?claimId ; :claimAmount ?claimAmount ;
           :incidentDate ?incidentDate ; :forPolicy ?policy ;
           :hasEvent ?deniedEvent .
    ?policy :policyId ?policyId .
    ?deniedEvent :eventType "denied" ; :eventTimestamp ?deniedDate .

    OPTIONAL {
        ?policy :hasCoverage ?cov .
        ?cov :coverageAmount ?coverageAmount ;
             :deductible ?deductible ;
             :validFrom ?covValidFrom .
        OPTIONAL { ?cov :validTo ?covValidTo }
    }
}
ORDER BY ?claimId ?covValidFrom
"""

rows = ontop_query(q5)
current_claim = None
for r in rows:
    if r['claimId'] != current_claim:
        current_claim = r['claimId']
        print(f"\n  {r['claimId']}  ${float(r['claimAmount']):>10,.2f}  Incident: {r['incidentDate'][:10]}  Denied: {r['deniedDate'][:10]}  Policy: {r['policyId']}")
        if r.get('coverageAmount'):
            print(f"  Coverage versions:")
        else:
            print(f"  Coverage: NONE — no coverage records found")
    if r.get('coverageAmount'):
        vt = r['covValidTo'][:10] if r.get('covValidTo') else 'present'
        print(f"    ${float(r['coverageAmount']):>10,.2f}  deductible ${float(r['deductible']):>8,.2f}  {r['covValidFrom'][:10]} → {vt}")

print(f"\n  ─── Diagnosis ───")
print(f"  C003: incident 2026-01-15 falls in gap between coverage v1 (→ 2025-12-31) and v2 (2026-02-01 →)")
print(f"  C004: $15,000 claim exceeds net coverage ($10,000 − $2,000 = $8,000)")
print(f"  C005: denied event exists — filed after 90-day window (see Module 4 SHACL results)")


→ Proceed to **05-NL-to-SPARQL**
